In [ ]:
"""
Test de géocodage via Nominatim self-hosted.

Principe :
- Lecture des données d'entreprise depuis le BRONZE (raw.job_offer) uniquement
- JOIN sur le GOLD (serving.job_offer) pour récupérer score_relevancy
- Sélection des N meilleures entreprises (dédupliquées)
- Géocodage via l'instance Nominatim locale (pas d'appel Google, coût nul)

Usage:
    python test_geocode_nominatim.py --limit 20 --min-score 6
"""

import argparse
import os
import time
from typing import Optional

import psycopg2
import psycopg2.extras
import requests
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
NOMINATIM_URL = os.getenv("NOMINATIM_URL", "http://localhost:8088")

# Politesse même en self-hosted : évite de saturer le VPS pendant l'import/usage
SLEEP_BETWEEN_CALLS = float(os.getenv("GEOCODE_SLEEP", "0.1"))


# ---------------------------------------------------------------- SQL

# NOTE: les données company/city/country viennent EXCLUSIVEMENT du bronze.
# Le gold n'est utilisé que pour rapatrier score_relevancy.
QUERY_TOP_COMPANIES = """
SELECT
    b.company,
    b.city,
    b.country,
    b.location_raw,
    COUNT(*)                     AS nb_offres,
    ROUND(AVG(g.score_relevancy)::numeric, 2) AS score_moyen,
    MAX(g.score_relevancy)       AS score_max
FROM raw.job_offer b
JOIN serving.job_offer g
    ON g.id_offer = b.id_job
WHERE b.company IS NOT NULL
  AND g.score_relevancy IS NOT NULL
  AND g.score_relevancy >= %(min_score)s
GROUP BY b.company, b.city, b.country, b.location_raw
ORDER BY score_max DESC, score_moyen DESC, nb_offres DESC
LIMIT %(limit)s;
"""


def fetch_top_companies(conn, limit: int, min_score: float) -> list[dict]:
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(QUERY_TOP_COMPANIES, {"limit": limit, "min_score": min_score})
        return [dict(row) for row in cur.fetchall()]


# ---------------------------------------------------------------- Géocodage


def build_query(company: str, city: Optional[str], country: Optional[str]) -> str:
    """Construit la requête textuelle envoyée à Nominatim."""
    parts = [p for p in (company, city, country) if p]
    return ", ".join(parts)


def geocode(query: str, timeout: int = 10) -> Optional[dict]:
    """Appelle l'instance Nominatim locale. Retourne None si aucun résultat."""
    try:
        r = requests.get(
            f"{NOMINATIM_URL}/search",
            params={
                "q": query,
                "format": "jsonv2",
                "limit": 1,
                "addressdetails": 1,
            },
            headers={"User-Agent": "InternshipLatam/1.0 (self-hosted test)"},
            timeout=timeout,
        )
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  [ERREUR RESEAU] {query} -> {e}")
        return None

    data = r.json()
    if not data:
        return None

    hit = data[0]
    return {
        "lat": float(hit["lat"]),
        "lon": float(hit["lon"]),
        "display_name": hit.get("display_name"),
        "osm_type": hit.get("osm_type"),
        "category": hit.get("category"),
        "importance": hit.get("importance"),
    }


def geocode_with_fallback(company: str, city: Optional[str], country: Optional[str]) -> dict:
    """
    Cascade : entreprise+ville+pays -> ville+pays.
    Le fallback évite de perdre totalement la localisation quand
    l'entreprise n'est pas présente dans OSM (fréquent pour les PME).
    """
    q_full = build_query(company, city, country)
    res = geocode(q_full)
    if res:
        return {**res, "match_level": "company", "query_used": q_full}

    q_city = build_query(None, city, country)
    if q_city:
        time.sleep(SLEEP_BETWEEN_CALLS)
        res = geocode(q_city)
        if res:
            return {**res, "match_level": "city", "query_used": q_city}

    return {
        "lat": None,
        "lon": None,
        "display_name": None,
        "osm_type": None,
        "category": None,
        "importance": None,
        "match_level": "none",
        "query_used": q_full,
    }


# ---------------------------------------------------------------- Main


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=20, help="Nombre d'entreprises à tester")
    parser.add_argument("--min-score", type=float, default=6.0, help="Score relevancy minimum")
    args = parser.parse_args()

    if not DATABASE_URL:
        raise SystemExit("DATABASE_URL manquant dans l'environnement.")

    # Vérification que Nominatim répond avant de taper en base
    try:
        ping = requests.get(f"{NOMINATIM_URL}/status", timeout=5)
        print(f"Nominatim : {NOMINATIM_URL} -> HTTP {ping.status_code}\n")
    except requests.RequestException as e:
        raise SystemExit(f"Nominatim injoignable sur {NOMINATIM_URL} : {e}")

    conn = psycopg2.connect(DATABASE_URL)
    try:
        companies = fetch_top_companies(conn, args.limit, args.min_score)
    finally:
        conn.close()

    if not companies:
        print(f"Aucune entreprise avec score_relevancy >= {args.min_score}")
        return

    print(f"{len(companies)} entreprises récupérées (score >= {args.min_score})\n")
    print("-" * 100)

    stats = {"company": 0, "city": 0, "none": 0}

    for i, row in enumerate(companies, 1):
        company = row["company"]
        city = row["city"]
        country = row["country"]

        print(f"[{i}/{len(companies)}] {company} — {city}, {country}")
        print(f"   score_max={row['score_max']} | score_moyen={row['score_moyen']} | offres={row['nb_offres']}")

        result = geocode_with_fallback(company, city, country)
        stats[result["match_level"]] += 1

        if result["lat"] is not None:
            print(f"   -> {result['lat']:.5f}, {result['lon']:.5f}  [{result['match_level']}]")
            print(f"      {result['display_name']}")
        else:
            print(f"   -> AUCUN RESULTAT (query: {result['query_used']})")

        print("-" * 100)
        time.sleep(SLEEP_BETWEEN_CALLS)

    total = len(companies)
    print("\nRESUME")
    print(f"  Match entreprise : {stats['company']}/{total} ({stats['company']/total*100:.1f}%)")
    print(f"  Match ville seule: {stats['city']}/{total} ({stats['city']/total*100:.1f}%)")
    print(f"  Aucun match      : {stats['none']}/{total} ({stats['none']/total*100:.1f}%)")
    print("\nCoût total : 0,00 € (instance self-hosted)")


if __name__ == "__main__":
    main()